In [ ]:
FILES={'reranker_inputs.py': '"""Qwen reranker inputs: preserve the classification suffix when truncating.\n\nMatches the official Qwen3-Reranker Transformers recipe: reserve prompt tokens,\ntruncate the body, append the untouched prefix/suffix, and left-pad the batch.\n"""\n\nQWEN_RERANK_PREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the \'\n                     \'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\')\nQWEN_RERANK_SUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef qwen_reranker_inputs(tokenizer, queries, documents, instruction, max_length, return_tensors=\'pt\'):\n    if len(queries) != len(documents) or not queries:\n        raise ValueError(\'Expected equal nonempty query/document batches\')\n    if tokenizer.padding_side != \'left\':\n        raise ValueError(\'Qwen last-position scoring requires left padding\')\n    prefix = tokenizer.encode(QWEN_RERANK_PREFIX, add_special_tokens=False)\n    suffix = tokenizer.encode(QWEN_RERANK_SUFFIX, add_special_tokens=False)\n    available = max_length - len(prefix) - len(suffix)\n    if available < 1:\n        raise ValueError(\'Token limit leaves no room for query/document body\')\n    bodies = [f\'<Instruct>: {instruction}\\n<Query>: {q}\\n<Document>: {d}\' for q,d in zip(queries,documents)]\n    encoded = tokenizer(bodies, padding=False, truncation=True, add_special_tokens=False,\n                        return_attention_mask=False, max_length=available)\n    ids = [prefix + body + suffix for body in encoded[\'input_ids\']]\n    assert all(len(row) <= max_length and row[-len(suffix):] == suffix for row in ids)\n    return tokenizer.pad({\'input_ids\': ids}, padding=True, return_tensors=return_tensors)\n', 'score_passage_corpus.py': '"""Kaggle GPU: score prepared parts with durable, hash-bound checkpoints."""\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoTokenizer,AutoModelForSequenceClassification,AutoModelForCausalLM\n\nfrom reranker_inputs import qwen_reranker_inputs,QWEN_RERANK_PREFIX,QWEN_RERANK_SUFFIX\nfrom assemble_passage_submission import assemble\n\nINSTRUCTION=\'Given a web search query, retrieve relevant passages that answer the query\'\nPAIR_TOKENS=4096\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef one(root,pattern):\n    hits=sorted(root.rglob(pattern))\n    if len(hits)!=1:raise ValueError(f\'{pattern}: expected one input, found {len(hits)}\')\n    return hits[0]\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model\',choices=[\'bge\',\'qwen\'],required=True)\n    parser.add_argument(\'--resume-root\',type=Path)\n    args=parser.parse_args();key=args.model\n    root=Path(\'/kaggle/input\');out=Path(\'/kaggle/working\');started=time.monotonic()\n    gpu=torch.cuda.get_device_name(0)\n    if \'6000\' not in gpu:raise ValueError(\'Expected RTX PRO6000, found \'+gpu)\n    print(\'GPU:\',gpu,flush=True)\n    manifest_path=one(root,\'passage_candidates_manifest.json\');candidate_root=manifest_path.parent\n    candidates=json.loads(manifest_path.read_text())\n    if not candidates[\'complete\'] or candidates[\'queries\']!=1200:raise ValueError(\'Incomplete full candidates\')\n    qpath=candidate_root/\'passage_queries.parquet\'\n    if sha(qpath)!=candidates[\'query_file_sha256\']:raise ValueError(\'Corrupted queries\')\n    queries=dict(pd.read_parquet(qpath,columns=[\'id\',\'query\']).itertuples(index=False,name=None))\n    name=\'bge-reranker-v2-m3\' if key==\'bge\' else \'Qwen3-Reranker-4B\'\n    model_path=one(root,name+\'/config.json\').parent\n    spec=dict(model=key,candidate_manifest_sha256=sha(manifest_path),model_config_sha256=sha(model_path/\'config.json\'),\n        implementation_sha256=sha(Path(__file__)),input_helper_sha256=sha(Path(__file__).with_name(\'reranker_inputs.py\')),\n        max_pair_tokens=PAIR_TOKENS,instruction=INSTRUCTION,precision=\'bfloat16\',attention=\'sdpa\')\n    report=dict(complete=False,gpu=gpu,spec=spec,parts=[],pairs=0,truncations=0)\n    report_path=out/f\'passage_scores_{key}_manifest.json\';destination=out/f\'passage_scores_{key}\';destination.mkdir(exist_ok=True)\n    if args.resume_root:\n        previous=json.loads((args.resume_root/report_path.name).read_text())\n        if previous[\'spec\']!=spec:raise ValueError(\'Resume scoring contract changed\')\n        for part in previous[\'parts\']:\n            source=args.resume_root/part[\'path\']\n            if sha(source)!=part[\'sha256\']:raise ValueError(\'Corrupted resume score\')\n            target=out/part[\'path\'];target.parent.mkdir(exist_ok=True)\n            shutil.copyfile(source,target);report[\'parts\'].append(part)\n    done={p[\'candidate_path\']:p for p in report[\'parts\']}\n    def checkpoint():\n        report[\'pairs\']=sum(p[\'pairs\'] for p in report[\'parts\'])\n        report[\'elapsed_seconds\']=time.monotonic()-started\n        temp=report_path.with_suffix(\'.tmp\');temp.write_text(json.dumps(report,indent=2));temp.replace(report_path)\n    checkpoint()\n    tok=AutoTokenizer.from_pretrained(model_path,local_files_only=True,padding_side=\'left\' if key==\'qwen\' else \'right\')\n    cls=AutoModelForSequenceClassification if key==\'bge\' else AutoModelForCausalLM\n    model=cls.from_pretrained(model_path,torch_dtype=torch.bfloat16,local_files_only=True,attn_implementation=\'sdpa\').cuda().eval()\n    if key==\'qwen\':\n        yes,no=tok.convert_tokens_to_ids(\'yes\'),tok.convert_tokens_to_ids(\'no\')\n        if yes==no or yes==tok.unk_token_id or no==tok.unk_token_id:raise ValueError(\'Invalid yes/no token IDs\')\n        overhead=len(tok.encode(QWEN_RERANK_PREFIX,add_special_tokens=False))+len(tok.encode(QWEN_RERANK_SUFFIX,add_special_tokens=False))\n    batch_size=32 if key==\'bge\' else 16\n    for part in candidates[\'parts\']:\n        if part[\'path\'] in done:continue\n        part_started=time.monotonic();source=candidate_root/part[\'path\']\n        if sha(source)!=part[\'sha256\']:raise ValueError(\'Corrupted candidate part\')\n        frame=pd.read_parquet(source)\n        if len(frame)!=part[\'pairs\'] or frame.passage_id.duplicated().any():raise ValueError(\'Bad part identity\')\n        texts=frame.chunk_text.tolist();qs=[str(queries[int(q)]) for q in frame.query_id]\n        order=np.argsort([len(t) for t in texts],kind=\'stable\');values=np.empty(len(frame),dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(order),batch_size):\n                ix=order[start:start+batch_size];q=[qs[i] for i in ix];d=[texts[i] for i in ix]\n                if key==\'bge\':\n                    encoded=tok(q,d,padding=True,truncation=False,return_tensors=\'pt\')\n                    if encoded[\'input_ids\'].shape[1]>PAIR_TOKENS:raise ValueError(\'BGE over context budget; no silent truncation\')\n                    result=torch.sigmoid(model(**encoded.to(\'cuda\')).logits.view(-1).float())\n                else:\n                    bodies=[f\'<Instruct>: {INSTRUCTION}\\n<Query>: {a}\\n<Document>: {b}\' for a,b in zip(q,d)]\n                    lengths=tok(bodies,add_special_tokens=False,truncation=False)[\'input_ids\']\n                    if any(len(ids)+overhead>PAIR_TOKENS for ids in lengths):raise ValueError(\'Qwen over context budget; no silent truncation\')\n                    encoded=qwen_reranker_inputs(tok,q,d,INSTRUCTION,PAIR_TOKENS).to(\'cuda\')\n                    logits=model(**encoded,logits_to_keep=1).logits[:,-1,:]\n                    result=torch.softmax(torch.stack([logits[:,no],logits[:,yes]],dim=1).float(),dim=1)[:,1]\n                values[ix]=result.cpu().numpy()\n                if start//batch_size%100==0:\n                    rate=(start+len(ix))/max(time.monotonic()-part_started,.01)\n                    print(key,source.name,start+len(ix),\'/\',len(order),\'pairs/s\',round(rate,1),flush=True)\n        if not np.isfinite(values).all():raise ValueError(\'Nonfinite model score\')\n        target=destination/source.name\n        pd.DataFrame(dict(passage_id=frame.passage_id,score=values)).to_parquet(target,index=False,compression=\'zstd\')\n        report[\'parts\'].append(dict(candidate_path=part[\'path\'],candidate_sha256=part[\'sha256\'],\n            path=str(target.relative_to(out)),sha256=sha(target),pairs=len(frame),seconds=time.monotonic()-part_started))\n        checkpoint();print(\'checkpoint\',key,len(report[\'parts\']),\'/\',len(candidates[\'parts\']),report[\'pairs\'],flush=True)\n    del model,tok;gc.collect();torch.cuda.empty_cache()\n    report[\'complete\']=True;checkpoint()\n    if report[\'pairs\']!=candidates[\'pairs\']:raise ValueError(\'Scoring coverage mismatch\')\n    result=assemble(candidate_root,out,key)\n    print(\'COMPLETE\',json.dumps(result),flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'assemble_passage_submission.py': '"""Select scored, source-exact passages and stream a DEFLATE submission ZIP."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef choose_query(frame,n_docs=200,n_chunks=100,cover_head=50,min_tokens=40):\n    """Max-passage doc ranking, cover the head, then score-order remaining windows.\n\n    Parameters are fixed across queries. No score-to-probability calibration.\n    """\n    if frame.passage_id.duplicated().any():raise ValueError(\'Duplicate passage ID\')\n    values=frame[\'score\'].to_numpy(dtype=float)\n    if not np.isfinite(values).all() or ((values<0)|(values>1)).any():raise ValueError(\'Invalid scores\')\n    ranked=frame[frame.passage_tokens>=min_tokens].sort_values(\n        [\'score\',\'seed_rank\',\'passage_id\'],ascending=[False,True,True],kind=\'stable\')\n    records=ranked.to_dict(\'records\');best={}\n    for row in records:\n        if row[\'end\']-row[\'start\']!=len(row[\'chunk_text\']):raise ValueError(\'Invalid passage offsets\')\n        for doc in row[\'doc_ids\']:best.setdefault(int(doc),row)\n    docs=list(best)[:n_docs];docset=set(docs);doc_rank={d:i for i,d in enumerate(docs)}\n    chosen=[];used=set();windows={};entries=set()\n    def add(row):\n        if row[\'passage_id\'] in used or len(chosen)>=n_chunks:return\n        start,end=int(row[\'start\']),int(row[\'end\']);key=row[\'text_sha256\']\n        for a,b in windows.get(key,[]):\n            overlap=max(0,min(end,b)-max(start,a))\n            if overlap/min(end-start,b-a)>=.8:return\n        ids=sorted((int(d) for d in row[\'doc_ids\'] if int(d) in docset),key=doc_rank.get)\n        added=False\n        for doc in ids:\n            identity=(doc,row[\'chunk_text\'])\n            if identity in entries:continue\n            if len(chosen)>=n_chunks:break\n            entries.add(identity);chosen.append(dict(doc_id=doc,chunk_text=row[\'chunk_text\']));added=True\n        if added:\n            used.add(row[\'passage_id\']);windows.setdefault(key,[]).append((start,end))\n    for doc in docs[:cover_head]:add(best[doc])\n    for row in records:add(row)\n    if not docs or not chosen:raise ValueError(\'No usable documents/passages\')\n    return docs,chosen\n\n\ndef assemble(candidate_root,score_root,model,n_docs=200,start_chunks=100,max_bytes=99_000_000):\n    candidate_root,score_root=map(Path,(candidate_root,score_root))\n    manifest_path=candidate_root/\'passage_candidates_manifest.json\'\n    candidates=json.loads(manifest_path.read_text())\n    score_manifest=json.loads((score_root/f\'passage_scores_{model}_manifest.json\').read_text())\n    if not candidates[\'complete\'] or not score_manifest[\'complete\']:raise ValueError(\'Incomplete inputs\')\n    if score_manifest[\'spec\'][\'candidate_manifest_sha256\']!=sha(manifest_path):raise ValueError(\'Wrong candidate version\')\n    parts={p[\'candidate_path\']:p for p in score_manifest[\'parts\']}\n    expected=set(candidates[\'spec\'][\'query_ids\']);counts=start_chunks\n    staging=score_root/f\'_submission_{model}\';staging.mkdir(exist_ok=True)\n    zip_path=score_root/f\'medxir-passage-{model}-d{n_docs}.zip\'\n    while counts>=20:\n        written=set();doc_count=chunk_count=covered=0\n        for part in candidates[\'parts\']:\n            source=candidate_root/part[\'path\'];scored_part=parts[part[\'path\']];scores=score_root/scored_part[\'path\']\n            if sha(source)!=part[\'sha256\'] or sha(scores)!=scored_part[\'sha256\']:raise ValueError(\'Corrupted input part\')\n            frame=pd.read_parquet(source).merge(pd.read_parquet(scores),on=\'passage_id\',validate=\'one_to_one\')\n            if len(frame)!=part[\'pairs\'] or set(frame.query_id)!=set(part[\'query_ids\']):raise ValueError(\'Part coverage mismatch\')\n            for qid,group in frame.groupby(\'query_id\',sort=False):\n                qid=int(qid)\n                if qid in written:raise ValueError(\'Duplicate output query\')\n                docs,chunks=choose_query(group,n_docs,counts)\n                entry=dict(id=qid,relevant_docs=docs,relevant_chunks=chunks)\n                (staging/f\'{qid}.json\').write_text(json.dumps(entry,ensure_ascii=False,separators=(\',\',\':\')),encoding=\'utf8\')\n                written.add(qid);doc_count+=len(docs);chunk_count+=len(chunks);covered+=len({c[\'doc_id\'] for c in chunks})\n        if written!=expected:raise ValueError(\'Submission query mismatch\')\n        with zipfile.ZipFile(zip_path,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:\n            with archive.open(\'submission.json\',\'w\') as handle:\n                handle.write(b\'[\')\n                for i,qid in enumerate(sorted(written)):\n                    if i:handle.write(b\',\')\n                    with (staging/f\'{qid}.json\').open(\'rb\') as item:shutil.copyfileobj(item,handle)\n                handle.write(b\']\')\n        print(\'ZIP\',model,\'cap\',counts,\'MB\',round(zip_path.stat().st_size/1e6,2),flush=True)\n        if zip_path.stat().st_size<max_bytes:break\n        counts=min(counts-1,int(counts*.95))\n    else:raise ValueError(\'Cannot fit submission under byte limit\')\n    with zipfile.ZipFile(zip_path) as archive:\n        infos=archive.infolist()\n        assert len(infos)==1 and infos[0].compress_type==zipfile.ZIP_DEFLATED\n        if archive.testzip() is not None:raise ValueError(\'ZIP CRC failure\')\n    for path in staging.glob(\'*.json\'):path.unlink()\n    staging.rmdir()\n    result=dict(complete=True,model=model,queries=len(written),zip_bytes=zip_path.stat().st_size,zip_sha256=sha(zip_path),\n        candidate_manifest_sha256=sha(manifest_path),score_manifest_sha256=sha(score_root/f\'passage_scores_{model}_manifest.json\'),\n        selection=dict(doc_depth=n_docs,chunk_cap=counts,cover_head=50,min_passage_tokens=40,near_overlap_threshold=.8),\n        docs_per_query=doc_count/len(written),chunks_per_query=chunk_count/len(written),covered_docs_per_query=covered/len(written),\n        source_fidelity=\'Text copied unchanged from hashed candidates; source offsets asserted during generation.\',\n        relevance_metrics=\'UNMEASURED: no gold labels\')\n    (score_root/f\'passage_{model}_submission_manifest.json\').write_text(json.dumps(result,indent=2))\n    return result\n'}
ARGS=['--model', 'bge']
from pathlib import Path
import sys,subprocess
for name,source in FILES.items():Path('/kaggle/working',name).write_text(source,encoding='utf8')
subprocess.run([sys.executable,'-u','/kaggle/working/score_passage_corpus.py',*ARGS],check=True)
